In [ ]:
from google_play_scraper import Sort, reviews
import pandas as pd
import time

In [ ]:
def fetch_reviews(app_id, score=None, lang='en', country='us',
                  sort=Sort.NEWEST, page_size=200, max_retries=5, pause=1.0):
    all_rows = []
    token = None

    # first page
    batch, token = reviews(app_id, lang=lang, country=country, sort=sort,
                           count=page_size, filter_score_with=score)
    all_rows.extend(batch)

    while token is not None and token.token is not None:
        for attempt in range(max_retries):
            try:
                batch, new_token = reviews(app_id, continuation_token=token)
            except Exception as e:
                print(f"  error: {e} (retry {attempt+1}/{max_retries})")
                time.sleep(pause * (attempt + 1))
                continue
            if batch:                      # got data -> move on
                break
            time.sleep(pause * (attempt + 1))  # empty page -> retry same token
        else:
            print(f"  gave up after {max_retries} empty/failed pages")
            break

        all_rows.extend(batch)
        token = new_token
        if len(all_rows) % 10000 < page_size:
            print(f"  score={score}: {len(all_rows):,} reviews so far")

    return all_rows


app_id = 'com.amazon.avod.thirdpartyclient'
frames = []
for score in [1, 2, 3, 4, 5]:
    print(f"Fetching {score}-star reviews...")
    rows = fetch_reviews(app_id, score=score)
    print(f"  -> {len(rows):,} reviews")
    frames.append(pd.DataFrame(rows))

df = pd.concat(frames, ignore_index=True).drop_duplicates(subset='reviewId')
print(f"Total unique reviews: {len(df):,}")

In [ ]:
df['at'] = pd.to_datetime(df['at'])
df['at'].dt.year.value_counts().sort_index()